# Music Annotation Project V5.1: Reconstruction Benchmark and Failure Analysis

**Runtime:** CPU. Select **Runtime > Run all**.

This corrected benchmark applies every reconstruction method to the same synthetic occlusion made from a clean page and its corrected annotation mask. It therefore compares reconstruction methods fairly, independently of YOLO detection quality.

Methods:
- OpenCV Telea;
- OpenCV Navier–Stokes;
- biharmonic inpainting;
- notation-aware hybrid reconstruction using an inpainting base plus explicit staff-line continuation.

The benchmark reports separate metrics for:
- all masked pixels;
- masked staff-line pixels;
- masked printed-symbol pixels;
- masked background-paper pixels.

It also creates enlarged component crops rather than relying on whole-page contact sheets.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas matplotlib scikit-image


## Configuration

The notation-aware method is executed here against the synthetic page. Do not load Notebook 9's real-page output into this synthetic benchmark: it was generated from a different observed image and may have a different filename namespace.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import cv2, json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim
from skimage.restoration import inpaint as sk_inpaint

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
CLEAN_DIR=ROOT/'source'/'clean'
MASK_DIR=ROOT/'masks'/'corrected'
OUTPUT_ROOT=ROOT/'reconstruction_benchmark'
RUN_NAME=datetime.now().strftime('%Y%m%d_%H%M%S')
OUT=OUTPUT_ROOT/RUN_NAME
TABLE_DIR=OUT/'tables'; PAGE_DIR=OUT/'page_comparisons'; CROP_DIR=OUT/'failure_crops'; MASK_CLASS_DIR=OUT/'mask_classes'
for folder in [OUT,TABLE_DIR,PAGE_DIR,CROP_DIR,MASK_CLASS_DIR]: folder.mkdir(parents=True,exist_ok=True)

SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
MAX_PAGE_SHEETS=20
MAX_FAILURE_CROPS=60
CROP_MARGIN=32
MIN_COMPONENT_AREA=20
OCCLUSION_VALUE=50
STAFF_KERNEL_WIDTH=35
STAFF_DILATE_X=7
STAFF_DILATE_Y=3
EDGE_TOLERANCE=1

# Base algorithm used inside the notation-aware hybrid.
# Change to 'navier_stokes' after the first run if its staff/symbol metrics are better.
HYBRID_BASE='telea'

def image_map(folder):
    return {p.stem:p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if folder.exists() else {}

clean_files=image_map(CLEAN_DIR); mask_files=image_map(MASK_DIR)
pages=sorted(set(clean_files)&set(mask_files))
if not pages: raise RuntimeError('No matching clean pages and corrected masks were found.')
print('Matched benchmark pages:',len(pages))
print('Output:',OUT)


## Detect staff-line, symbol and background evaluation regions

The class masks are derived only from clean ground truth:

- **staff** is long horizontal printed ink detected morphologically;
- **symbol** is other printed ink, with a small exclusion band around staff lines;
- **background** is paper inside the annotation mask.

These are evaluation strata, not perfect optical-music-recognition labels. Inspect the saved class-mask overlays before treating symbol scores as definitive.


In [ ]:
def printed_ink(clean):
    # Adaptive threshold is more tolerant of page tint and faded scans than one global threshold.
    return cv2.adaptiveThreshold(clean,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0

def classify_mask_regions(clean,annotation_mask):
    ink=printed_ink(clean).astype(np.uint8)*255
    horizontal=cv2.morphologyEx(ink,cv2.MORPH_OPEN,np.ones((1,STAFF_KERNEL_WIDTH),np.uint8))
    staff=cv2.dilate(horizontal,np.ones((STAFF_DILATE_Y,STAFF_DILATE_X),np.uint8))>0
    # Protect staff neighbourhood from being counted as a symbol merely due to crossings.
    staff_neighbourhood=cv2.dilate(staff.astype(np.uint8),np.ones((5,9),np.uint8))>0
    symbol=(ink>0)&~staff_neighbourhood
    mask=annotation_mask>0
    return {
        'all':mask,
        'staff':mask&staff,
        'symbol':mask&symbol,
        'background':mask&~(ink>0),
    },staff,ink>0

def save_class_overlay(page,clean,regions):
    colour=cv2.cvtColor(clean,cv2.COLOR_GRAY2BGR)
    layer=colour.copy()
    layer[regions['background']]=(255,180,0)
    layer[regions['staff']]=(0,180,0)
    layer[regions['symbol']]=(0,0,255)
    cv2.imwrite(str(MASK_CLASS_DIR/f'{page}_classes.png'),cv2.addWeighted(colour,.55,layer,.45,0))


## Reconstruction methods

The notation-aware hybrid deliberately separates two tasks:

1. a generic inpainting base fills paper texture and small gaps;
2. detected staff-line segments are continued through the masked interval only when supported by visible ink immediately to the left or right.

Biharmonic inpainting is included as a third non-semantic baseline. It can be useful for smooth paper/background regions, but should not be expected to infer missing musical semantics.


In [ ]:
def telea(observed,mask): return cv2.inpaint(observed,mask,3,cv2.INPAINT_TELEA)
def navier_stokes(observed,mask): return cv2.inpaint(observed,mask,5,cv2.INPAINT_NS)

def biharmonic(observed,mask):
    result=sk_inpaint.inpaint_biharmonic(observed.astype(np.float32)/255.0,mask>0,channel_axis=None)
    return np.clip(result*255,0,255).astype(np.uint8)

def estimate_staff_rows(clean,mask):
    visible=clean.copy();visible[mask>0]=255
    ink=(visible<190).astype(np.uint8)
    opened=cv2.morphologyEx(ink,cv2.MORPH_OPEN,np.ones((1,STAFF_KERNEL_WIDTH),np.uint8))
    profile=opened.sum(axis=1).astype(np.float32)
    smooth=cv2.GaussianBlur(profile.reshape(-1,1),(1,5),0).ravel()
    threshold=max(12,np.percentile(smooth,85))
    rows=[]
    for y in range(1,len(smooth)-1):
        if smooth[y]>=threshold and smooth[y]>=smooth[y-1] and smooth[y]>=smooth[y+1]:
            if not rows or y-rows[-1]>2: rows.append(y)
            elif smooth[y]>smooth[rows[-1]]: rows[-1]=y
    return rows

def supported_line_intensity(clean,mask,y,x,radius=14):
    h,w=clean.shape
    values=[]
    for yy in range(max(0,y-1),min(h,y+2)):
        lx0=max(0,x-radius); lx1=x
        rx0=x+1; rx1=min(w,x+radius+1)
        if lx1>lx0:
            v=clean[yy,lx0:lx1][mask[yy,lx0:lx1]==0]; values.extend(v[v<190].tolist())
        if rx1>rx0:
            v=clean[yy,rx0:rx1][mask[yy,rx0:rx1]==0]; values.extend(v[v<190].tolist())
    return int(np.percentile(values,30)) if values else None

def continue_staff_lines(base,clean,mask,rows):
    result=base.copy();h,w=clean.shape
    for y in rows:
        xs=np.where(mask[max(0,y-1):min(h,y+2)].max(axis=0)>0)[0]
        for x in xs:
            value=supported_line_intensity(clean,mask,y,x)
            if value is not None:
                result[max(0,y-1):min(h,y+2),x]=min(value,135)
    return result

def notation_aware(observed,mask,clean_visible_reference):
    base=telea(observed,mask) if HYBRID_BASE=='telea' else navier_stokes(observed,mask)
    rows=estimate_staff_rows(clean_visible_reference,mask)
    return continue_staff_lines(base,clean_visible_reference,mask,rows)

def reconstruct_methods(observed,mask,clean):
    return {
        'telea':telea(observed,mask),
        'navier_stokes':navier_stokes(observed,mask),
        'biharmonic':biharmonic(observed,mask),
        'notation_aware':notation_aware(observed,mask,clean),
    }


## Region-specific metrics

`edge_f1_tolerant` allows a one-pixel localisation tolerance, which is appropriate for thin scanned notation. Exact edge F1 is retained as a stricter diagnostic.

For staff and symbol strata, the most useful measures are:

- ink recall: how much ground-truth printed ink reappeared;
- ink precision: how much reconstructed dark ink corresponds to ground-truth ink;
- tolerant edge F1: whether structural boundaries reappear near the correct location.


In [ ]:
def edge_map(image): return cv2.Canny(image,70,170)>0

def binary_f1(reference,prediction,region,tolerance=0):
    ref=reference&region; pred=prediction&region
    if tolerance>0:
        k=np.ones((2*tolerance+1,2*tolerance+1),np.uint8)
        ref_d=cv2.dilate(ref.astype(np.uint8),k)>0
        pred_d=cv2.dilate(pred.astype(np.uint8),k)>0
        precision=float(np.sum(pred&ref_d)/max(np.sum(pred),1))
        recall=float(np.sum(ref&pred_d)/max(np.sum(ref),1))
    else:
        tp=np.sum(ref&pred);precision=float(tp/max(np.sum(pred),1));recall=float(tp/max(np.sum(ref),1))
    f1=2*precision*recall/max(precision+recall,1e-12)
    return precision,recall,f1

def region_metrics(clean,reconstructed,region):
    if not np.any(region): return None
    c=clean.astype(np.float32);r=reconstructed.astype(np.float32)
    diff=np.abs(c-r);mse=float(np.mean((c[region]-r[region])**2))
    mae=float(diff[region].mean());psnr=float(20*np.log10(255.0/math.sqrt(max(mse,1e-9))))
    _,ssim_map=ssim(clean,reconstructed,data_range=255,full=True)
    gt_ink=printed_ink(clean); pred_ink=printed_ink(reconstructed)
    ink_precision,ink_recall,ink_f1=binary_f1(gt_ink,pred_ink,region,0)
    exact_precision,exact_recall,edge_f1=binary_f1(edge_map(clean),edge_map(reconstructed),region,0)
    tol_precision,tol_recall,edge_f1_tolerant=binary_f1(edge_map(clean),edge_map(reconstructed),region,EDGE_TOLERANCE)
    return {'pixels':int(region.sum()),'mae':mae,'psnr':psnr,'ssim':float(ssim_map[region].mean()),'ink_precision':ink_precision,'ink_recall':ink_recall,'ink_f1':ink_f1,'edge_f1':edge_f1,'edge_f1_tolerant':edge_f1_tolerant}

def outside_mask_drift(observed,reconstructed,mask):
    outside=mask==0
    return float(np.abs(observed.astype(np.float32)-reconstructed.astype(np.float32))[outside].mean()) if np.any(outside) else 0.0


## Run the controlled benchmark


In [ ]:
rows=[];cache={};skipped=[]
for page in pages:
    clean=cv2.imread(str(clean_files[page]),cv2.IMREAD_GRAYSCALE)
    mask=cv2.imread(str(mask_files[page]),cv2.IMREAD_GRAYSCALE)
    if clean is None or mask is None or clean.shape!=mask.shape:
        skipped.append((page,'unreadable or shape mismatch'));continue
    mask=np.where(mask>0,255,0).astype(np.uint8)
    if not np.any(mask):
        skipped.append((page,'empty mask'));continue
    observed=clean.copy();observed[mask>0]=OCCLUSION_VALUE
    regions,staff_map,ink_map=classify_mask_regions(clean,mask)
    save_class_overlay(page,clean,regions)
    methods=reconstruct_methods(observed,mask,clean)
    cache[page]={'clean':clean,'observed':observed,'mask':mask,'regions':regions,'methods':methods}
    for method,reconstructed in methods.items():
        for region_name,region in regions.items():
            metric=region_metrics(clean,reconstructed,region)
            if metric is None: continue
            rows.append({'page':page,'method':method,'region':region_name,'outside_mask_drift':outside_mask_drift(observed,reconstructed,mask),**metric})
    print('Benchmarked',page)

results=pd.DataFrame(rows)
if results.empty: raise RuntimeError('No pages could be benchmarked.')
results.to_csv(TABLE_DIR/'reconstruction_metrics_by_region.csv',index=False)
summary=(results.groupby(['method','region'],as_index=False)
         .agg(pages=('page','nunique'),pixels=('pixels','sum'),mae=('mae','mean'),psnr=('psnr','mean'),ssim=('ssim','mean'),ink_precision=('ink_precision','mean'),ink_recall=('ink_recall','mean'),ink_f1=('ink_f1','mean'),edge_f1=('edge_f1','mean'),edge_f1_tolerant=('edge_f1_tolerant','mean'),outside_mask_drift=('outside_mask_drift','mean')))
summary.to_csv(TABLE_DIR/'reconstruction_summary_by_region.csv',index=False)
all_summary=summary[summary.region=='all'].copy()
all_summary.to_csv(OUT/'reconstruction_summary.csv',index=False)
print('Skipped:',skipped)
display(summary.sort_values(['region','edge_f1_tolerant'],ascending=[True,False]))


## Whole-page comparison sheets

These are useful for gross artefacts only. Fine reconstruction quality should be judged from the component crops below.


In [ ]:
method_order=['telea','navier_stokes','biharmonic','notation_aware']
for page,data in list(cache.items())[:MAX_PAGE_SHEETS]:
    panels=[data['clean'],data['observed']]+[data['methods'][m] for m in method_order]
    names=['clean ground truth','synthetic occlusion']+method_order
    fig,axes=plt.subplots(1,len(panels),figsize=(5*len(panels),8))
    for ax,image,name in zip(axes,panels,names):
        ax.imshow(image,cmap='gray',vmin=0,vmax=255);ax.set_title(name);ax.axis('off')
    fig.suptitle(page);plt.tight_layout();plt.savefig(PAGE_DIR/f'{page}_comparison.png',dpi=150,bbox_inches='tight');plt.close(fig)
print('Page comparisons:',PAGE_DIR)


## Zoomed component-level failure crops

Each crop is centred on one connected annotation component and contains:

`ground truth | occlusion | Telea | Navier–Stokes | biharmonic | notation-aware | error heatmap`

The crops are ranked by the worst notation-aware tolerant edge F1. The error heatmap displays absolute error for the notation-aware method.


In [ ]:
component_rows=[]
for page,data in cache.items():
    mask=data['mask'];n,labels,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8)
    for component in range(1,n):
        x,y,w,h,area=stats[component]
        if area<MIN_COMPONENT_AREA: continue
        component_region=labels==component
        metric=region_metrics(data['clean'],data['methods']['notation_aware'],component_region)
        if metric:
            component_rows.append({'page':page,'component':component,'x':int(x),'y':int(y),'w':int(w),'h':int(h),'area':int(area),**metric})
component_df=pd.DataFrame(component_rows).sort_values(['edge_f1_tolerant','ink_f1','area'],ascending=[True,True,False])
component_df.to_csv(TABLE_DIR/'notation_aware_component_failures.csv',index=False)

for rank,row in component_df.head(MAX_FAILURE_CROPS).reset_index(drop=True).iterrows():
    data=cache[row.page];h0,w0=data['clean'].shape
    x0=max(0,int(row.x)-CROP_MARGIN);y0=max(0,int(row.y)-CROP_MARGIN);x1=min(w0,int(row.x+row.w)+CROP_MARGIN);y1=min(h0,int(row.y+row.h)+CROP_MARGIN)
    panels=[data['clean'][y0:y1,x0:x1],data['observed'][y0:y1,x0:x1]]+[data['methods'][m][y0:y1,x0:x1] for m in method_order]
    error=np.abs(data['clean'][y0:y1,x0:x1].astype(np.int16)-data['methods']['notation_aware'][y0:y1,x0:x1].astype(np.int16)).astype(np.uint8)
    names=['ground truth','occlusion']+method_order+['notation-aware error']
    fig,axes=plt.subplots(1,7,figsize=(24,4))
    for ax,image,name in zip(axes,panels+[error],names):
        ax.imshow(image,cmap='inferno' if name.endswith('error') else 'gray',vmin=0,vmax=255);ax.set_title(name);ax.axis('off')
    fig.suptitle(f"rank {rank+1} | {row.page} component {int(row.component)} | tolerant edge F1={row.edge_f1_tolerant:.3f} | ink F1={row.ink_f1:.3f}")
    plt.tight_layout();plt.savefig(CROP_DIR/f'{rank+1:03d}_{row.page}_component_{int(row.component):03d}.png',dpi=180,bbox_inches='tight');plt.close(fig)
print('Failure crops:',CROP_DIR)


## Direct method comparison and automatic verdict

The decision rule prioritises notation structure:

1. symbol tolerant edge F1;
2. staff ink recall;
3. all-region SSIM;
4. outside-mask drift as a safety check.

The notebook reports measured deltas but does not declare practical significance from an arbitrary universal threshold.


In [ ]:
pivot=summary.pivot(index='method',columns='region',values=['edge_f1_tolerant','ink_recall','ink_f1','ssim','mae','outside_mask_drift'])
pivot.to_csv(TABLE_DIR/'method_metric_matrix.csv')
display(pivot)

baseline_methods=['telea','navier_stokes','biharmonic']
score_rows=[]
for method in summary.method.unique():
    def value(region,metric):
        hit=summary[(summary.method==method)&(summary.region==region)]
        return float(hit.iloc[0][metric]) if not hit.empty else np.nan
    score_rows.append({'method':method,'symbol_edge_f1_tolerant':value('symbol','edge_f1_tolerant'),'staff_ink_recall':value('staff','ink_recall'),'all_ssim':value('all','ssim'),'all_mae':value('all','mae'),'outside_mask_drift':value('all','outside_mask_drift')})
score_df=pd.DataFrame(score_rows)
score_df.to_csv(TABLE_DIR/'decision_metrics.csv',index=False)
display(score_df.sort_values(['symbol_edge_f1_tolerant','staff_ink_recall'],ascending=False))

ours=score_df[score_df.method=='notation_aware'].iloc[0]
best_symbol=score_df[score_df.method.isin(baseline_methods)].sort_values('symbol_edge_f1_tolerant',ascending=False).iloc[0]
best_staff=score_df[score_df.method.isin(baseline_methods)].sort_values('staff_ink_recall',ascending=False).iloc[0]
verdict={
 'notation_aware_symbol_edge_delta_vs_best_baseline':float(ours.symbol_edge_f1_tolerant-best_symbol.symbol_edge_f1_tolerant),
 'best_symbol_baseline':best_symbol.method,
 'notation_aware_staff_recall_delta_vs_best_baseline':float(ours.staff_ink_recall-best_staff.staff_ink_recall),
 'best_staff_baseline':best_staff.method,
 'outside_mask_drift':float(ours.outside_mask_drift),
 'interpretation':'Positive structural deltas support the notation-aware method. Mixed deltas mean routing staff and symbol regions to different methods is preferable.'
}
(OUT/'benchmark_verdict.json').write_text(json.dumps(verdict,indent=2))
print(json.dumps(verdict,indent=2))


## Outputs

The most important files are:

- `reconstruction_summary.csv`
- `tables/reconstruction_summary_by_region.csv`
- `tables/decision_metrics.csv`
- `tables/notation_aware_component_failures.csv`
- `failure_crops/`
- `benchmark_verdict.json`

Before trusting the separate staff/symbol rankings, inspect several files in `mask_classes/`. Red should mostly cover non-staff printed symbols, green staff lines, and blue background paper inside the annotation mask.


In [ ]:
print('Run:',OUT)
for path in [OUT/'reconstruction_summary.csv',TABLE_DIR/'reconstruction_summary_by_region.csv',TABLE_DIR/'decision_metrics.csv',TABLE_DIR/'notation_aware_component_failures.csv',OUT/'benchmark_verdict.json']:
    print(path)
print('Class overlays:',MASK_CLASS_DIR)
print('Failure crops:',CROP_DIR)
